In [ ]:
"""
combine_places.py
=================
Steps:
  1. Unzip all zip files from the 3 Google Places API folders
  2. Combine JSON files per city (excluding *_consolidated* files) → json_folder/
  3. Combine all city JSONs into ALL_CITIES.json
  4. Flatten ALL_CITIES.json into ALL_CITIES.csv

Usage (in Colab):
  !python combine_places.py
"""

import json
import os
import re
import shutil
import zipfile
from pathlib import Path

import pandas as pd

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────

SOURCE_FOLDERS = [
    "/content/drive/MyDrive/Upwork/patrick-bwn-project/Google Places API",
    "/content/drive/MyDrive/Upwork/patrick-bwn-project/Google Places API - Part 2",
    "/content/drive/MyDrive/Upwork/patrick-bwn-project/Google Places API - Part 3",
]

JSON_FOLDER  = Path("json_folder")        # per-city combined JSONs
EXTRACT_DIR  = Path("_extracted")         # temp unzip workspace
FINAL_JSON   = Path("ALL_CITIES.json")
FINAL_CSV    = Path("ALL_CITIES.csv")


# ─────────────────────────────────────────────────────────────
# HELPER: derive city slug from zip filename
# ─────────────────────────────────────────────────────────────

def city_slug(zip_path: Path) -> str:
    """
    'austin-places.zip'  →  'austin'
    'austin_places.zip'  →  'austin'
    'New_York_places.zip'→  'new_york'
    """
    stem = zip_path.stem                     # e.g. "austin-places"
    slug = re.sub(r"[-_]places$", "", stem, flags=re.IGNORECASE)
    return slug.lower().replace("-", "_")


# ─────────────────────────────────────────────────────────────
# STEP 1 & 2 — Unzip + combine per city
# ─────────────────────────────────────────────────────────────

def step1_2_unzip_and_combine() -> None:
    print("\n── STEP 1 & 2: Unzip + combine JSON files per city ──")

    JSON_FOLDER.mkdir(exist_ok=True)
    EXTRACT_DIR.mkdir(exist_ok=True)

    # city_slug → list of place-records (dicts)
    city_records: dict[str, list] = {}

    for folder in SOURCE_FOLDERS:
        folder_path = Path(folder)
        if not folder_path.exists():
            print(f"  ⚠  Folder not found, skipping: {folder}")
            continue

        zip_files = sorted(folder_path.glob("*.zip"))
        if not zip_files:
            print(f"  ⚠  No zip files in: {folder}")
            continue

        print(f"\n  📂 {folder}  ({len(zip_files)} zip file(s))")

        for zip_path in zip_files:
            city = city_slug(zip_path)
            extract_to = EXTRACT_DIR / zip_path.stem

            # Unzip
            try:
                with zipfile.ZipFile(zip_path, "r") as zf:
                    zf.extractall(extract_to)
                print(f"    ✔ Unzipped: {zip_path.name}  →  {city}")
            except zipfile.BadZipFile:
                print(f"    ✗ Bad zip, skipping: {zip_path.name}")
                continue

            # Collect JSON files (exclude *_consolidated*)
            json_files = [
                f for f in extract_to.rglob("*.json")
                if "_consolidated" not in f.name.lower()
            ]

            records_before = len(city_records.get(city, []))
            seen_ids: set = set()

            if city not in city_records:
                city_records[city] = []

            for jf in json_files:
                try:
                    raw = json.loads(jf.read_text(encoding="utf-8"))
                except Exception as exc:
                    print(f"      ⚠ Could not read {jf.name}: {exc}")
                    continue

                # Each JSON page file has a "places" key (raw API response)
                # OR it could already be a list of flattened records
                if isinstance(raw, dict) and "places" in raw:
                    places = raw["places"]
                elif isinstance(raw, list):
                    places = raw
                else:
                    places = [raw]

                for place in places:
                    pid = place.get("id") or place.get("place_id")
                    if pid and pid in seen_ids:
                        continue
                    if pid:
                        seen_ids.add(pid)
                    city_records[city].append(place)

            added = len(city_records[city]) - records_before
            print(f"      {len(json_files)} JSON file(s) → {added} records merged")

    # Write per-city combined JSONs
    print("\n  💾 Writing per-city JSON files …")
    for city, records in city_records.items():
        out_path = JSON_FOLDER / f"{city}.json"
        out_path.write_text(
            json.dumps(records, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        print(f"    {out_path}  ({len(records):,} records)")

    print(f"\n  ✅ Step 1 & 2 complete — {len(city_records)} city file(s) in '{JSON_FOLDER}/'")


# ─────────────────────────────────────────────────────────────
# STEP 3 — Combine all city JSONs → ALL_CITIES.json
# ─────────────────────────────────────────────────────────────

def step3_combine_all() -> list:
    print("\n── STEP 3: Combine all city JSONs into ALL_CITIES.json ──")

    all_records: list = []
    seen_ids: set = set()

    city_files = sorted(JSON_FOLDER.glob("*.json"))
    if not city_files:
        print("  ✗ No city JSON files found in json_folder/. Run steps 1-2 first.")
        return []

    for city_file in city_files:
        records = json.loads(city_file.read_text(encoding="utf-8"))
        before = len(all_records)
        for rec in records:
            pid = rec.get("id") or rec.get("place_id")
            if pid and pid in seen_ids:
                continue
            if pid:
                seen_ids.add(pid)
            all_records.append(rec)
        print(f"  {city_file.name:<40} +{len(all_records) - before:,}")

    FINAL_JSON.write_text(
        json.dumps(all_records, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"\n  ✅ Step 3 complete — {len(all_records):,} unique records → '{FINAL_JSON}'")
    return all_records


# ─────────────────────────────────────────────────────────────
# STEP 4 — Flatten helpers (from scraping script)
# ─────────────────────────────────────────────────────────────

def _extract_address_component(components: list, target_type: str) -> str | None:
    for comp in components:
        if target_type in comp.get("types", []):
            return comp.get("longText")
    return None


def _convert_photo_url(url: str | None) -> str | None:
    if not url:
        return None
    match = re.search(r'!1s([^!]+)!2e10', url)
    if not match:
        return None
    image_id = match.group(1)
    if image_id.startswith(("AF1Qip", "CIHM0og", "CIABIh")):
        return f"https://lh5.googleusercontent.com/p/{image_id}=s1600"
    return None


def _flatten_place(place: dict) -> dict:
    """
    Handles BOTH raw Google Places API records (with 'id', 'displayName', etc.)
    AND already-processed records (with 'place_id', 'business_name', etc.).
    """
    # ── Detect record format ───────────────────────────────────
    is_raw = "displayName" in place or ("id" in place and "place_id" not in place)

    if is_raw:
        # Raw Google Places API response shape
        comps     = place.get("addressComponents", [])
        photos_in = place.get("photos", [])
        photo_urls = [
            u for u in (
                _convert_photo_url(p.get("googleMapsUri"))
                for p in photos_in
            ) if u
        ]
        return {
            "city_search":    place.get("city", ""),
            "zone":           place.get("zone", ""),
            "query":          place.get("query", ""),
            "place_id":        place.get("id", ""),
            "resource_name":   place.get("name", ""),
            "business_name":   place.get("displayName", {}).get("text", ""),
            "business_status": place.get("businessStatus", ""),
            "primary_type":              place.get("primaryType", ""),
            "primary_type_display_name": place.get("primaryTypeDisplayName", {}).get("text", ""),
            "all_types":                 "|".join(place.get("types", [])),
            "latitude":  place.get("location", {}).get("latitude"),
            "longitude": place.get("location", {}).get("longitude"),
            "formatted_address":       place.get("formattedAddress", ""),
            "short_formatted_address": place.get("shortFormattedAddress", ""),
            "neighborhood": _extract_address_component(comps, "neighborhood"),
            "city":         _extract_address_component(comps, "locality"),
            "county":       _extract_address_component(comps, "administrative_area_level_2"),
            "state":        _extract_address_component(comps, "administrative_area_level_1"),
            "country":      _extract_address_component(comps, "country"),
            "postal_code":  _extract_address_component(comps, "postal_code"),
            "phone":   place.get("nationalPhoneNumber", ""),
            "website": place.get("websiteUri", ""),
            "email":   "",
            "instagram_url": "",
            "facebook_url":  "",
            "linkedin_url":  "",
            "tiktok_url":    "",
            "youtube_url":   "",
            "twitter_x_url": "",
            "rating":          place.get("rating"),
            "review_count":    place.get("userRatingCount"),
            "source_platform": "Google Places API",
            "hours": "|".join(
                place.get("regularOpeningHours", {}).get("weekdayDescriptions", [])
            ),
            "price_level": place.get("priceLevel", ""),
            "price_range": place.get("priceRange", ""),
            "google_maps_uri": place.get("googleMapsUri", ""),
            "directions_uri":  place.get("googleMapsLinks", {}).get("directionsUri", ""),
            "place_uri":       place.get("googleMapsLinks", {}).get("placeUri", ""),
            "reviews_uri":     place.get("googleMapsLinks", {}).get("reviewsUri", ""),
            "wheelchair_accessible_entrance":
                place.get("accessibilityOptions", {}).get("wheelchairAccessibleEntrance"),
            "wheelchair_accessible_restroom":
                place.get("accessibilityOptions", {}).get("wheelchairAccessibleRestroom"),
            "utc_offset_minutes":    place.get("utcOffsetMinutes"),
            "icon_background_color": place.get("iconBackgroundColor", ""),
            "icon_mask_base_uri":    place.get("iconMaskBaseUri", ""),
            "photo_count": len(photo_urls),
            "photo_urls":  "|".join(photo_urls),
        }

    else:
        # Already-processed record shape (from build_record / _flatten_place output)
        comps     = place.get("address_components", [])
        photos_in = place.get("photos_references", [])
        photo_urls = [
            u for u in (
                _convert_photo_url(p.get("googleMapsUri"))
                for p in photos_in
            ) if u
        ]
        return {
            "city_search":    place.get("city", ""),
            "zone":           place.get("zone", ""),
            "query":          place.get("query", ""),
            "place_id":        place.get("place_id", ""),
            "resource_name":   place.get("resource_name", ""),
            "business_name":   place.get("business_name", ""),
            "business_status": place.get("business_status", ""),
            "primary_type":              place.get("primary_type", ""),
            "primary_type_display_name": place.get("primary_type_display_name", ""),
            "all_types":                 "|".join(place.get("all_types", [])),
            "latitude":  place.get("latitude"),
            "longitude": place.get("longitude"),
            "formatted_address":       place.get("formatted_address", ""),
            "short_formatted_address": place.get("short_formatted_address", ""),
            "neighborhood": _extract_address_component(comps, "neighborhood"),
            "city":         _extract_address_component(comps, "locality"),
            "county":       _extract_address_component(comps, "administrative_area_level_2"),
            "state":        _extract_address_component(comps, "administrative_area_level_1"),
            "country":      _extract_address_component(comps, "country"),
            "postal_code":  _extract_address_component(comps, "postal_code"),
            "phone":   place.get("phone", ""),
            "website": place.get("website", ""),
            "email":   "",
            "instagram_url": "",
            "facebook_url":  "",
            "linkedin_url":  "",
            "tiktok_url":    "",
            "youtube_url":   "",
            "twitter_x_url": "",
            "rating":          place.get("rating"),
            "review_count":    place.get("user_rating_count"),
            "source_platform": "Google Places API",
            "hours": "|".join(
                place.get("regular_opening_hours", {}).get("weekdayDescriptions", [])
            ) if isinstance(place.get("regular_opening_hours"), dict) else "",
            "price_level": place.get("price_level", ""),
            "price_range": str(place.get("price_range", "")),
            "google_maps_uri": place.get("google_maps_uri", ""),
            "directions_uri":  place.get("google_maps_links", {}).get("directionsUri", "")
                               if isinstance(place.get("google_maps_links"), dict) else "",
            "place_uri":       place.get("google_maps_links", {}).get("placeUri", "")
                               if isinstance(place.get("google_maps_links"), dict) else "",
            "reviews_uri":     place.get("google_maps_links", {}).get("reviewsUri", "")
                               if isinstance(place.get("google_maps_links"), dict) else "",
            "wheelchair_accessible_entrance":
                place.get("accessibility_options", {}).get("wheelchairAccessibleEntrance")
                if isinstance(place.get("accessibility_options"), dict) else None,
            "wheelchair_accessible_restroom":
                place.get("accessibility_options", {}).get("wheelchairAccessibleRestroom")
                if isinstance(place.get("accessibility_options"), dict) else None,
            "utc_offset_minutes":    place.get("utc_offset_minutes"),
            "icon_background_color": place.get("icon_background_color", ""),
            "icon_mask_base_uri":    place.get("icon_mask_base_uri", ""),
            "photo_count": len(photo_urls),
            "photo_urls":  "|".join(photo_urls),
        }


def step4_flatten_to_csv(all_records: list) -> None:
    print("\n── STEP 4: Flatten to CSV ──")

    if not all_records:
        # Load from file if not passed directly
        if not FINAL_JSON.exists():
            print(f"  ✗ {FINAL_JSON} not found. Run step 3 first.")
            return
        all_records = json.loads(FINAL_JSON.read_text(encoding="utf-8"))

    rows = []
    errors = 0
    for rec in all_records:
        try:
            rows.append(_flatten_place(rec))
        except Exception as exc:
            errors += 1
            if errors <= 5:
                print(f"  ⚠ Flatten error (showing first 5): {exc}")

    df = pd.DataFrame(rows)
    df.to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")

    print(f"  ✅ Step 4 complete — {len(rows):,} rows → '{FINAL_CSV}'")
    if errors:
        print(f"  ⚠  {errors} record(s) failed to flatten and were skipped.")
    print(f"\n  Columns ({len(df.columns)}): {', '.join(df.columns.tolist())}")


# ─────────────────────────────────────────────────────────────
# CLEANUP
# ─────────────────────────────────────────────────────────────

def cleanup_temp() -> None:
    """Remove the temporary extraction directory."""
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
        print(f"\n  🗑  Removed temp dir: {EXTRACT_DIR}")


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    step1_2_unzip_and_combine()
    all_records = step3_combine_all()
    step4_flatten_to_csv(all_records)
    cleanup_temp()

    print("\n" + "═" * 55)
    print("  ALL DONE")
    print(f"  Per-city JSONs : {JSON_FOLDER}/")
    print(f"  Master JSON    : {FINAL_JSON}")
    print(f"  Final CSV      : {FINAL_CSV}")
    print("═" * 55)


── STEP 1 & 2: Unzip + combine JSON files per city ──

  📂 /content/drive/MyDrive/Upwork/patrick-bwn-project/Google Places API  (25 zip file(s))
    ✔ Unzipped: austin-places.zip  →  austin
      533 JSON file(s) → 1851 records merged
    ✔ Unzipped: boston_places.zip  →  boston
      597 JSON file(s) → 1723 records merged
    ✔ Unzipped: charlotte-places.zip  →  charlotte
      717 JSON file(s) → 1952 records merged
    ✔ Unzipped: chicago-places.zip  →  chicago
      744 JSON file(s) → 3261 records merged
    ✔ Unzipped: columbus-places.zip  →  columbus
      729 JSON file(s) → 1585 records merged
    ✔ Unzipped: dallas-places.zip  →  dallas
      746 JSON file(s) → 2163 records merged
    ✔ Unzipped: denver-places.zip  →  denver
      629 JSON file(s) → 1916 records merged
    ✔ Unzipped: el_paso_places.zip  →  el_paso
      626 JSON file(s) → 1347 records merged
    ✔ Unzipped: fort-worth-places.zip  →  fort_worth
      749 JSON file(s) → 1843 records merged
    ✔ Unzipped: housto